# Chuẩn bị lại 3 bộ dữ liệu độc lập, chia fold theo bệnh nhân (v2)

Notebook này thay thế phần chuẩn bị dữ liệu của `lung_download_label_5fold.ipynb`
theo góp ý chỉnh sửa. Thay vì 3 bộ `gse68465 / tcga_luad / tcga_lusc` trộn mẫu
Normal với mẫu u, notebook xuất **3 bộ dữ liệu độc lập, mỗi bộ cho đúng một tác vụ**:

| Bộ dữ liệu | Tác vụ | Mẫu được giữ |
|---|---|---|
| `gse68465_stage_tumor_only` | Phân loại giai đoạn (I/II/III/IV) | Chỉ mẫu u LUAD có giai đoạn hợp lệ; **loại vĩnh viễn 19 mẫu RNA tham chiếu Stratagene** |
| `tcga_luad_lusc_primary_tumor` | Phân loại LUAD vs LUSC | Chỉ mẫu *Primary Tumor* của TCGA-LUAD và TCGA-LUSC, 1 mẫu / bệnh nhân |
| `tcga_normal_vs_tumor_sanity` | Kiểm tra (sanity check) Normal vs u | Chỉ *Solid Tissue Normal* và *Primary Tumor* của TCGA-LUAD/LUSC |

Mỗi bộ được chia thành 5 phần P1–P5 bằng `StratifiedGroupKFold` với
`groups = patient_id` (mọi mẫu của một bệnh nhân luôn nằm trong cùng một phần),
rồi xoay vòng thành 5 fold: fold *r* có test = P*r*, val = P*r+1* (P5 → P1),
train = 3 phần còn lại.

**Mỗi bộ gồm** (trong `<OUTDIR>/<bo>.zip`, các file nhỏ được chép thêm ra `<OUTDIR>/<bo>/`):

| File | Nội dung |
|---|---|
| `expression.h5` (key `expression`) | Ma trận biểu hiện mẫu × gene (float32), index `sample_id` |
| `metadata.csv` | Metadata đã lọc của các mẫu được giữ (có `patient_id`, `class`, `label`) |
| `fold_assignment.csv` | `sample_id, patient_id, class, label, part, fold_1..fold_5` (train/val/test) |
| `class_stats.csv` | Số bệnh nhân và số mẫu theo lớp, trong từng phần và từng tập của từng fold |
| `filter_log.csv` | Số mẫu / số bệnh nhân trước và sau từng bước lọc |
| `config.json` | Seed, seed được chọn, commit hash, tiêu chí lọc, ánh xạ nhãn, phiên bản thư viện |
| `DATA_CARD.md` | Data card: tác vụ, tiêu chí lọc mẫu, số mẫu, cách chia, hạn chế |
| `folds/fold_<r>/{train,val,test}.h5` | (tuỳ chọn, `EXPORT_FOLD_FILES`) bảng mẫu × (gene..., `label`) cho các notebook FS/FE cũ |

Notebook chạy trên Colab (cần Drive). Tiền xử lý biểu hiện giữ nguyên như bản cũ:
GSE68465 là log2 MAS5, probe → gene; TCGA là log2(TPM+1), chỉ gene `protein_coding`.
**Không chuẩn hoá (scaling)** dữ liệu ở bước này.

## 1. Thiết lập

Điền `GIT_COMMIT` bằng kết quả `git rev-parse HEAD` của repo KLTN tại thời điểm
chạy (Colab không có thư mục `.git`). Nếu để trống, notebook tự tìm repo trong
`REPO_DIRS`; không tìm được thì ghi `"unknown"` và in cảnh báo.

In [ ]:
import os, io, re, sys, json, time, gzip, tarfile, zipfile, shutil, platform, subprocess, warnings
import urllib.request
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import StratifiedGroupKFold

warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKDIR = "/content/kltn_data"                              # du lieu tho + file tam (local)
    OUTDIR = "/content/drive/MyDrive/KLTN/data/datasets_v2"     # 3 bo du lieu ket qua
else:
    WORKDIR = os.path.abspath("kltn_data")
    OUTDIR = os.path.join(WORKDIR, "datasets_v2")

RAWDIR = os.path.join(WORKDIR, "raw")
STAGEDIR = os.path.join(WORKDIR, "datasets_v2_tmp")   # dung tung bo truoc khi zip
for d in (RAWDIR, STAGEDIR, OUTDIR):
    os.makedirs(d, exist_ok=True)

NOTEBOOK = "notebooks/00_data/lung_datasets_v2_patient_folds.ipynb"

# --- Tai lap ---
SEED = 42                                       # seed goc
SEED_SEARCH = list(range(SEED, SEED + 100))     # thu 100 seed, chon cach chia can bang lop nhat
N_FOLDS = 5
# Xoay vong: fold r -> test = phan r, val = phan r+1 (5 -> 1), train = 3 phan con lai
FOLD_PLAN = {r: {"test": r, "val": r % N_FOLDS + 1,
                 "train": [p for p in range(1, N_FOLDS + 1) if p not in (r, r % N_FOLDS + 1)]}
             for r in range(1, N_FOLDS + 1)}

# Commit cua repo KLTN. Colab khong co .git -> dan ket qua `git rev-parse HEAD` vao day.
GIT_COMMIT = ""
REPO_DIRS = [os.getcwd(), os.path.abspath(os.path.join(os.getcwd(), "..", "..")),
             "/content/drive/MyDrive/KLTN", "/content/KLTN"]

# --- Xuat file ---
EXPORT_FOLD_FILES = True      # them folds/fold_<r>/{train,val,test}.<fmt> cho cac notebook FS/FE cu
FOLD_FILE_FORMATS = ["h5"]    # them "csv" neu can (zip lon hon nhieu)

print("Colab:", IN_COLAB)
print("Du lieu tho:", RAWDIR)
print("Ket qua    :", OUTDIR)
for r, plan in FOLD_PLAN.items():
    print(f"  fold {r}: train = phan {plan['train']}, val = phan {plan['val']}, test = phan {plan['test']}")

In [ ]:
def get_commit_hash():
    if GIT_COMMIT.strip():
        return GIT_COMMIT.strip(), "nhap tay (GIT_COMMIT)"
    for d in REPO_DIRS:
        if not os.path.isdir(d):
            continue
        try:
            h = subprocess.check_output(["git", "-C", d, "rev-parse", "HEAD"],
                                        text=True, stderr=subprocess.DEVNULL).strip()
        except Exception:
            continue
        dirty = subprocess.run(["git", "-C", d, "status", "--porcelain"],
                               capture_output=True, text=True).stdout.strip()
        return h + ("-dirty" if dirty else ""), f"git ({d})"
    return "unknown", "khong tim thay repo git"


COMMIT_HASH, COMMIT_SOURCE = get_commit_hash()
RUN_STARTED_UTC = datetime.now(timezone.utc).isoformat(timespec="seconds")
VERSIONS = {"python": platform.python_version(), "numpy": np.__version__,
            "pandas": pd.__version__, "scikit-learn": sklearn.__version__}

print("Commit:", COMMIT_HASH, f"({COMMIT_SOURCE})")
if COMMIT_HASH == "unknown":
    print("CANH BAO: chua co commit hash -> dien GIT_COMMIT o o tren de config.json tai lap duoc")
print("Bat dau:", RUN_STARTED_UTC, "|", VERSIONS)

In [ ]:
%pip install -q tables

## 2. Hàm dùng chung

- Chuẩn hoá nhãn giai đoạn: giữ nguyên logic cũ (đọc `Stage IIIB`, `stage: 2`, mã pTNM
  kiểu `pN0pT1` → stage AJCC giả định M0). Chỉ `I/II/III/IV` là hợp lệ.
- `collapse_by_symbol`: nhiều probe/Ensembl ID cùng 1 gene → giữ đặc trưng có phương sai
  lớn nhất (không dùng nhãn).
- `FilterLog`: ghi số mẫu / số bệnh nhân sau mỗi bước lọc, dùng cho `filter_log.csv`,
  `config.json` và data card.

In [ ]:
ARABIC_TO_ROMAN = {"1": "I", "2": "II", "3": "III", "4": "IV"}
MISSING = {"", "na", "n/a", "nan", "none", "null", "unknown",
           "not available", "not reported", "---", "-", "."}
VALID_STAGES = ["I", "II", "III", "IV"]


def stage_from_tnm_string(s):
    # Quy tac AJCC NSCLC, gia dinh M0 (cohort da phau thuat cat bo):
    #   N2/N3, hoac T4, hoac (T3 va N1) -> III ; T3 hoac N1 -> II ; con lai -> I
    t_match = re.search(r"p?t\s*([0-4])", s)
    n_match = re.search(r"p?n\s*([0-3])", s)
    if not t_match or not n_match:
        return None
    t, n = int(t_match.group(1)), int(n_match.group(1))
    if n >= 2 or t >= 4 or (t == 3 and n == 1):
        return "III"
    if t == 3 or n == 1:
        return "II"
    return "I"


def normalize_stage(x):
    # 'pathological stage: IIIB' -> 'III', 'disease_stage: pN0pT1' -> 'I'. None neu khong doc duoc.
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    s = str(x).strip().lower()
    if s in MISSING:
        return None
    # ma pTNM phai bat truoc, neu khong doan re.sub duoi se xoa mat pT/pN
    if re.search(r"p?t\s*[0-4]", s) and re.search(r"p?n\s*[0-3]", s):
        tnm_stage = stage_from_tnm_string(s)
        if tnm_stage is not None:
            return tnm_stage
    s = re.sub(r"^.*stage\s*[:=]?\s*", "", s).strip()
    if s in MISSING:
        return None
    if re.match(r"0(?![0-9])", s):
        return "0"
    m = re.match(r"(iv|iii|ii|i)(?![ivx])", s)   # khong dung \b vi "ia" se khong khop
    if m:
        return m.group(1).upper()
    m = re.match(r"([1-4])(?![0-9])", s)
    if m:
        return ARABIC_TO_ROMAN[m.group(1)]
    return None


def collapse_by_symbol(expr, symbols):
    # expr: feature x sample; symbols: Series cung index. Nhieu feature cung 1 gene
    # -> giu feature co phuong sai lon nhat (khong dung nhan, khong ro ri).
    sym = symbols.reindex(expr.index).dropna()
    e = expr.loc[sym.index]
    order = e.var(axis=1).sort_values(ascending=False).index
    e, sym = e.loc[order], sym.loc[order]
    keep = ~sym.duplicated().values
    e = e[keep]
    e.index = pd.Index(sym[keep].values, name="gene")
    return e.sort_index()


class FilterLog:
    # So mau / so benh nhan sau tung buoc loc -> filter_log.csv, config.json, DATA_CARD.md
    def __init__(self, name):
        self.name, self.rows = name, []

    def add(self, step, criterion, df):
        prev = self.rows[-1]["n_samples"] if self.rows else len(df)
        row = {"step": len(self.rows), "buoc": step, "tieu_chi": criterion,
               "n_samples": int(len(df)), "n_patients": int(df["patient_id"].nunique()),
               "n_removed": int(prev - len(df))}
        self.rows.append(row)
        print(f"  [{self.name}] {step}: {row['n_samples']} mau / {row['n_patients']} benh nhan"
              f" (loai {row['n_removed']})")

    def table(self):
        return pd.DataFrame(self.rows)


def md_table(df):
    head = "| " + " | ".join(map(str, df.columns)) + " |"
    sep = "|" + "|".join(["---"] * df.shape[1]) + "|"
    body = ["| " + " | ".join(str(v) for v in row) + " |" for row in df.itertuples(index=False)]
    return "\n".join([head, sep] + body)

## 3. GSE68465 → `gse68465_stage_tumor_only`

Tải `series_matrix.txt.gz`, tách metadata và ma trận probe × mẫu.

In [ ]:
GSE = "GSE68465"
GSE_RAW = os.path.join(RAWDIR, GSE)
os.makedirs(GSE_RAW, exist_ok=True)


def download(url, dest_dir):
    path = os.path.join(dest_dir, url.rsplit("/", 1)[-1])
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    urllib.request.urlretrieve(url, path)
    return path


def geo_stub(acc):
    # Thu muc FTP cua GEO: GSE68465 -> GSE68nnn, GPL96 -> GPLnnn
    prefix, num = acc[:3], acc[3:]
    return prefix + (num[:-3] if len(num) > 3 else "") + "nnn"


def matrix_file_urls(gse):
    stub = geo_stub(gse)
    base = f"https://ftp.ncbi.nlm.nih.gov/geo/series/{stub}/{gse}/matrix/"
    with urllib.request.urlopen(base, timeout=120) as r:
        html = r.read().decode("utf-8", errors="replace")
    files = sorted(set(re.findall(r'href="([^"]*series_matrix\.txt\.gz)"', html)))
    if not files:
        raise RuntimeError(f"Khong tim thay series matrix cho {gse}")
    return [base + f for f in files]


def parse_series_matrix(path):
    # Tra ve (expr, pheno). expr: probe x sample. pheno: sample x metadata.
    meta, table_lines, in_table = {}, [], False
    with gzip.open(path, "rt", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.rstrip("\n").rstrip("\r")
            if line.startswith("!series_matrix_table_begin"):
                in_table = True
                continue
            if line.startswith("!series_matrix_table_end"):
                in_table = False
                continue
            if in_table:
                if line:
                    table_lines.append(line)
            elif line.startswith("!Sample_"):
                parts = line.split("\t")
                meta.setdefault(parts[0][1:], []).append([v.strip().strip('"') for v in parts[1:]])

    # key lap lai (vd Sample_characteristics_ch1) -> danh so
    flat = {}
    for key, occ in meta.items():
        if len(occ) == 1:
            flat[key] = occ[0]
        else:
            for i, o in enumerate(occ, 1):
                flat[f"{key}_{i}"] = o
    pheno = pd.DataFrame(flat).set_index("Sample_geo_accession")
    pheno.index.name = "gsm"

    expr = pd.read_csv(io.StringIO("\n".join(table_lines)), sep="\t", index_col=0, low_memory=False)
    expr.index.name = "probe"
    expr = expr.apply(pd.to_numeric, errors="coerce")
    expr = expr.loc[:, [c for c in expr.columns if c in pheno.index]]
    return expr, pheno


GSE_URLS = matrix_file_urls(GSE)
exprs, phenos_gse = [], []
for url in GSE_URLS:
    print("tai", url.rsplit("/", 1)[-1], "...", end=" ")
    e, p = parse_series_matrix(download(url, GSE_RAW))
    exprs.append(e)
    phenos_gse.append(p)
    print(f"{e.shape[0]} probe x {e.shape[1]} mau")

gse_expr = pd.concat(exprs, axis=1)
gse_pheno = pd.concat(phenos_gse, axis=0)
print("TONG:", gse_expr.shape, "| platform:", sorted(gse_pheno["Sample_platform_id"].unique()))

### 3.1. Lọc mẫu GSE68465

Thứ tự lọc (mỗi bước được ghi vào `filter_log.csv`):

1. **Loại vĩnh viễn mẫu RNA tham chiếu Stratagene** (`source_name` chứa `Stratagene`).
   Đây là 19 mẫu mà bản cũ gán nhãn `Normal`, nhưng thực chất là RNA thương mại dùng
   làm chuẩn tham chiếu, không phải mô phổi thường của bệnh nhân. Notebook `assert`
   đúng 19 mẫu bị loại; nếu GEO thay đổi số mẫu, notebook dừng để kiểm tra lại.
   Bước này không có tuỳ chọn tắt.
2. **Chỉ giữ mẫu u LUAD**: `disease_state` chứa `Adenocarcinoma`. Sau bước này không
   còn mẫu nào mang nhãn Normal.
3. **Chỉ giữ mẫu có giai đoạn hợp lệ** `I/II/III/IV` (đọc từ `disease_stage`, dạng pTNM).
   Mẫu không đọc được giai đoạn hoặc là giai đoạn 0 bị loại.

GSE68465 không có mã bệnh nhân riêng (mỗi mẫu u là một bệnh nhân), nên
`patient_id = GSM` trừ khi metadata có trường mã bệnh nhân/người hiến.

In [ ]:
EXPECTED_STRATAGENE = 19


def find_col(pheno, pattern):
    # Cot characteristics dau tien co gia tri khop pattern (vd 'disease_stage:')
    for c in pheno.columns:
        if pheno[c].astype(str).str.contains(pattern, case=False, na=False).any():
            return c
    return None


def after_colon(s):
    return s.astype(str).str.split(":", n=1).str[-1].str.strip()


STAGE_COL = find_col(gse_pheno, r"disease_stage\s*:")
STATE_COL = find_col(gse_pheno, r"disease_state\s*:")
PID_COL = find_col(gse_pheno, r"(patient|donor|subject)\s*(id|number|no)\.?\s*:")
SOURCE_COL = "Sample_source_name_ch1"
print("Cot stage:", STAGE_COL, "| disease_state:", STATE_COL, "| ma benh nhan:", PID_COL)

ph = gse_pheno.copy()
ph["patient_id"] = after_colon(ph[PID_COL]) if PID_COL else ph.index.to_series()
ph["source_name"] = ph[SOURCE_COL].astype(str)
ph["disease_state"] = after_colon(ph[STATE_COL])
ph["stage_raw"] = after_colon(ph[STAGE_COL])
ph["stage"] = ph[STAGE_COL].map(normalize_stage)

# Cac truong characteristics con lai (sex, age, ...) -> cot char_<ten>, giu trong metadata.csv
for c in [c for c in ph.columns if c.startswith("Sample_characteristics")]:
    key = ph[c].astype(str).str.split(":", n=1).str[0].str.strip().mode().iloc[0]
    name = "char_" + re.sub(r"[^0-9a-z]+", "_", key.lower()).strip("_")
    if name not in ph.columns:
        ph[name] = after_colon(ph[c])

print("\nsource_name x disease_state TRUOC khi loc:")
display(pd.crosstab(ph["source_name"], ph["disease_state"]))

gse_log = FilterLog("gse68465_stage_tumor_only")
gse_log.add("0_series_matrix", f"Tat ca mau trong series matrix {GSE} (GPL96)", ph)

# 1. Loai vinh vien RNA tham chieu Stratagene
is_ref = ph["source_name"].str.contains("stratagene", case=False, na=False)
print(f"\nMau Stratagene: {int(is_ref.sum())} | disease_state cua chung:",
      ph.loc[is_ref, "disease_state"].value_counts().to_dict())
assert int(is_ref.sum()) == EXPECTED_STRATAGENE, \
    f"Mong doi {EXPECTED_STRATAGENE} mau Stratagene, tim thay {int(is_ref.sum())} -> kiem tra lai GEO"
GSE_REMOVED_REFERENCE = sorted(ph.index[is_ref])
ph = ph.loc[~is_ref]
gse_log.add("1_loai_stratagene",
            "Loai vinh vien 19 mau RNA tham chieu Stratagene (source_name chua 'Stratagene'), "
            "truoc day bi gan nham nhan Normal", ph)

# 2. Chi giu mau u LUAD
is_luad = (ph["disease_state"].str.contains("adenocarcinoma", case=False, na=False)
           & ~ph["disease_state"].str.contains("normal", case=False, na=False))
assert is_luad.any(), f"Khong co mau nao khop 'adenocarcinoma': {ph['disease_state'].value_counts().to_dict()}"
print("Loai (khong phai u LUAD):", ph.loc[~is_luad, "disease_state"].value_counts().to_dict())
ph = ph.loc[is_luad]
gse_log.add("2_chi_u_luad", "Chi giu mau u: disease_state chua 'Adenocarcinoma' (khong con mau Normal)", ph)

# 3. Chi giu giai doan hop le
valid = ph["stage"].isin(VALID_STAGES)
print("Loai (giai doan khong hop le):", ph.loc[~valid, "stage_raw"].value_counts(dropna=False).to_dict())
ph = ph.loc[valid]
gse_log.add("3_giai_doan_hop_le",
            "Chi giu mau co giai doan I/II/III/IV doc duoc tu disease_stage (pTNM, gia dinh M0); "
            "loai mau thieu giai doan hoac giai doan 0", ph)

# Kiem tra sau loc
assert not ph["source_name"].str.contains("stratagene", case=False).any()
assert not ph["disease_state"].str.contains("normal", case=False).any()
assert ph["stage"].isin(VALID_STAGES).all()

GSE_STAGE_LABELS = {"I": 0, "II": 1, "III": 2, "IV": 3}
ph["class"] = ph["stage"]
ph["label"] = ph["stage"].map(GSE_STAGE_LABELS).astype(int)
print("\nPhan bo lop sau loc:", ph["class"].value_counts().reindex(VALID_STAGES).fillna(0).astype(int).to_dict())
display(gse_log.table())

In [ ]:
# Thang do: MAS5 o thang tuyen tinh -> log2
e = gse_expr.loc[:, ph.index]
GSE_LOG2_APPLIED = bool(np.nanmax(e.values) >= 100)
if GSE_LOG2_APPLIED:
    e = np.log2(e.clip(lower=0) + 1)
    print("Da chuyen sang log2")

# Probe -> gene symbol tu GPLxx.annot.gz
GSE_GPL = ph["Sample_platform_id"].iloc[0]
GSE_ANNOT_URL = (f"https://ftp.ncbi.nlm.nih.gov/geo/platforms/{geo_stub(GSE_GPL)}/{GSE_GPL}"
                 f"/annot/{GSE_GPL}.annot.gz")
annot_path = download(GSE_ANNOT_URL, GSE_RAW)
lines, in_table = [], False
with gzip.open(annot_path, "rt", encoding="utf-8", errors="replace") as f:
    for line in f:
        if line.startswith("!platform_table_begin"):
            in_table = True
            continue
        if line.startswith("!platform_table_end"):
            break
        if in_table:
            lines.append(line)
annot = pd.read_csv(io.StringIO("".join(lines)), sep="\t", dtype=str)
sym = annot.set_index("ID")["Gene symbol"].str.strip()
sym = sym[sym.notna() & (sym != "") & ~sym.str.contains("///", regex=False)]
print(f"{GSE_GPL}: {len(sym)} probe anh xa duoc toi dung 1 gene")

n_probe_nan = int(e.isna().any(axis=1).sum())
e = e.dropna(axis=0, how="any")   # probe co NaN (neu co) bi bo
gse_X = collapse_by_symbol(e, sym).T.astype(np.float32)
gse_X.index.name, gse_X.columns.name = "sample_id", None

GSE_META_COLS = (["patient_id", "class", "label", "stage", "stage_raw", "disease_state", "source_name",
                  "Sample_platform_id"] + sorted(c for c in ph.columns if c.startswith("char_")))
gse_meta = ph.loc[gse_X.index, list(dict.fromkeys(GSE_META_COLS))].copy()
gse_meta.index.name = "sample_id"
print(f"Ma tran: {gse_X.shape} (mau x gene) | bo {n_probe_nan} probe co NaN")

## 4. TCGA-LUAD và TCGA-LUSC (GDC API, STAR - Counts)

Tải toàn bộ file STAR - Counts của 2 project một lần, dùng chung cho 2 bộ TCGA.
Khác bản cũ: biểu hiện được lưu theo `file_id` (không ghi đè khi một mẫu có nhiều
file) và metadata thêm `primary_diagnosis`. `patient_id = case submitter_id`
(`TCGA-XX-XXXX`), có đối chiếu với 12 ký tự đầu của barcode mẫu.

**Giai đoạn và `primary_diagnosis`:** một case GDC có thể có nhiều bản ghi `diagnoses` (u chỉ điểm,
*Prior primary*, *Synchronous primary*, *metastasis*...) theo thứ tự không cố định. Bản v2 đầu tiên chỉ đọc
`diagnoses[0]` nên mất giai đoạn ở 121 LUAD / 85 LUSC; metadata đã dựng được vá bằng
`scripts/patch_v2_tcga_stage.py`. Từ nay `select_diagnosis` dùng đúng quy tắc của script đó:
ứng viên = bản ghi có `ajcc_pathologic_stage` khác rỗng / `Not Reported`; cùng một giá trị → lấy;
khác nhau → `stage_conflict = True` và lấy bản ghi u chỉ điểm (`diagnosis_is_primary_disease = True`).
`primary_diagnosis` lấy từ bản ghi u chỉ điểm.

In [ ]:
PROJECTS = ["TCGA-LUAD", "TCGA-LUSC"]
API = "https://api.gdc.cancer.gov"
BATCH = 60   # so file tai moi lan; giam neu bi timeout


def _post(url, payload, timeout=600, retries=4, raw=False):
    req = urllib.request.Request(url, data=json.dumps(payload).encode(),
                                 headers={"Content-Type": "application/json"})
    last = None
    for attempt in range(retries):
        try:
            with urllib.request.urlopen(req, timeout=timeout) as r:
                return r.read() if raw else json.loads(r.read().decode())
        except Exception as ex:
            last = ex
            wait = 5 * (attempt + 1)
            print(f"    loi ({ex}), thu lai sau {wait}s ...")
            time.sleep(wait)
    raise RuntimeError(f"That bai sau {retries} lan: {last}")


# Lay MOI ban ghi diagnoses cua case (khong chi diagnoses[0]) -- xem select_diagnosis
DIAG_FIELDS = ["submitter_id", "ajcc_pathologic_stage", "ajcc_staging_system_edition",
               "ajcc_pathologic_t", "ajcc_pathologic_n", "ajcc_pathologic_m", "primary_diagnosis",
               "diagnosis_is_primary_disease", "classification_of_tumor"]
FIELDS = ",".join([
    "file_id", "file_name", "cases.submitter_id",
    "cases.samples.submitter_id", "cases.samples.sample_type", "cases.samples.is_ffpe",
] + [f"cases.diagnoses.{f}" for f in DIAG_FIELDS])


def query_files(project):
    filters = {"op": "and", "content": [
        {"op": "in", "content": {"field": "cases.project.project_id", "value": [project]}},
        {"op": "in", "content": {"field": "data_category", "value": ["Transcriptome Profiling"]}},
        {"op": "in", "content": {"field": "data_type", "value": ["Gene Expression Quantification"]}},
        {"op": "in", "content": {"field": "analysis.workflow_type", "value": ["STAR - Counts"]}},
    ]}
    res = _post(f"{API}/files", {"filters": filters, "fields": FIELDS,
                                 "format": "JSON", "size": "20000"})
    return res["data"]["hits"]


# Cung quy tac voi scripts/patch_v2_tcga_stage.py (script da va metadata cua lan dung dau tien).
# Ban cu chi doc diagnoses[0] -> mat giai doan khi ban ghi dau tien la Prior/Synchronous primary, metastasis...
STAGE_RE = re.compile(r"^Stage (IV|III|II|I)[ABC]?$")   # Stage IA/IB -> I, IIIC -> III, IVB -> IV


def _has_stage(d):
    s = d.get("ajcc_pathologic_stage")
    return s is not None and str(s).strip() not in ("", "Not Reported")


def select_diagnosis(diagnoses):
    # Tra ve (ban ghi chon hoac None, stage_conflict, rule):
    #  ung vien = ban ghi co ajcc_pathologic_stage khac rong / 'Not Reported'; khong co -> 'none';
    #  cung mot gia tri -> lay (uu tien ban ghi u chi diem de lay T/N/M, edition) -> 'single';
    #  khac nhau -> stage_conflict, lay ban ghi u chi diem neu co dung mot -> 'conflict_index_tumor'.
    cands = [d for d in diagnoses if _has_stage(d)]
    if not cands:
        return None, False, "none"
    index = [d for d in cands if d.get("diagnosis_is_primary_disease") is True]
    if len({d["ajcc_pathologic_stage"] for d in cands}) == 1:
        return (index or cands)[0], False, "single"
    if len(index) == 1:
        return index[0], True, "conflict_index_tumor"
    return None, True, "conflict_unresolved"


def stage_group(stage):
    m = STAGE_RE.match(str(stage).strip()) if stage is not None else None
    return m.group(1) if m else None


def index_tumor_diagnosis(diagnoses):
    index = [d for d in diagnoses if d.get("diagnosis_is_primary_disease") is True]
    return index[0] if len(index) == 1 else {}


def diagnosis_mismatch(project, diag):
    # LUAD ma chan doan te bao vay, hoac LUSC ma chan doan khong phai te bao vay
    if not diag:
        return False
    squamous = "squamous" in diag.lower() and "adenosquamous" not in diag.lower()
    return squamous if project == "TCGA-LUAD" else not squamous


def flatten(hits, project):
    rows = []
    for h in hits:
        case = (h.get("cases") or [{}])[0]
        samp = (case.get("samples") or [{}])[0]
        diags = case.get("diagnoses") or []
        chosen, conflict, rule = select_diagnosis(diags)
        c = chosen or {}
        pdx = index_tumor_diagnosis(diags).get("primary_diagnosis")
        rows.append({
            "file_id": h["file_id"],
            "case_id": case.get("submitter_id"),
            "sample_id": samp.get("submitter_id"),
            "sample_type": samp.get("sample_type"),
            "is_ffpe": samp.get("is_ffpe"),
            "primary_diagnosis": pdx,
            "stage_raw": c.get("ajcc_pathologic_stage"),
            "stage_group": stage_group(c.get("ajcc_pathologic_stage")),
            "ajcc_edition": c.get("ajcc_staging_system_edition"),
            "path_t": c.get("ajcc_pathologic_t"),
            "path_n": c.get("ajcc_pathologic_n"),
            "path_m": c.get("ajcc_pathologic_m"),
            "stage_conflict": conflict,
            "stage_rule": rule,
            "stage_diagnosis_submitter_id": c.get("submitter_id"),
            "diagnosis_mismatch": diagnosis_mismatch(project, pdx),
        })
    return pd.DataFrame(rows)


def make_batches(ids, batch_size):
    # GDC /data tra ve file THO (khong phai tar.gz) neu lo chi co 1 file_id
    # -> chia deu sao cho khong lo nao chi con 1 phan tu.
    n = len(ids)
    if n == 0:
        return []
    n_batches = (n + batch_size - 1) // batch_size
    if n_batches >= 2 and n % batch_size == 1:
        n_batches -= 1
    batches, start = [], 0
    for i in range(n_batches):
        size = -(-(n - start) // (n_batches - i))
        batches.append(ids[start:start + size])
        start += size
    return batches


def download_batch(file_ids, dest):
    # Cache chi hop le neu dung la gzip (magic bytes \x1f\x8b)
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        with open(dest, "rb") as f:
            if f.read(2) == b"\x1f\x8b":
                return dest
        print(f"    {os.path.basename(dest)} khong phai gzip hop le -- tai lai")
    blob = _post(f"{API}/data", {"ids": file_ids}, raw=True)
    with open(dest, "wb") as f:
        f.write(blob)
    return dest


def assemble(project, manifest):
    # Doc cac tar.gz -> (tpm: ensembl x file_id, gene_meta)
    wanted = set(manifest["file_id"])
    pdir = os.path.join(RAWDIR, project)
    tpm, gene_meta = {}, None
    for t in sorted(f for f in os.listdir(pdir) if f.endswith(".tar.gz")):
        with tarfile.open(os.path.join(pdir, t), "r:gz") as tar:
            for m in tar.getmembers():
                if not m.isfile() or not m.name.endswith(".tsv"):
                    continue
                fid = m.name.split("/")[0]
                if fid not in wanted:
                    continue
                df = pd.read_csv(io.BytesIO(tar.extractfile(m).read()), sep="\t",
                                 comment="#", low_memory=False)
                df = df[~df["gene_id"].astype(str).str.startswith("N_")].set_index("gene_id")
                if gene_meta is None:
                    gene_meta = df[["gene_name", "gene_type"]].copy()
                tpm[fid] = df["tpm_unstranded"].astype(np.float32)
    return pd.DataFrame(tpm), gene_meta


try:
    with urllib.request.urlopen(f"{API}/status", timeout=60) as r:
        GDC_STATUS = json.loads(r.read().decode())
except Exception as ex:
    GDC_STATUS = {"error": str(ex)}
print("GDC:", {k: GDC_STATUS.get(k) for k in ("data_release", "tag", "version", "error") if k in GDC_STATUS})

In [ ]:
tcga_raw = {}
for p in PROJECTS:
    print(f"\n=== {p} ===")
    mf = flatten(query_files(p), p)
    print(f"  {len(mf)} file |", mf["sample_type"].value_counts().to_dict())
    print("  stage_rule:", mf.drop_duplicates("case_id")["stage_rule"].value_counts().to_dict())
    bad = mf[mf["stage_raw"].notna() & mf["stage_group"].isna()]
    assert bad.empty, f"ajcc_pathologic_stage khong anh xa duoc ve I-IV: {bad['stage_raw'].unique()}"
    assert not (mf["stage_rule"] == "conflict_unresolved").any(), "co case conflict khong co dung 1 u chi diem"

    pdir = os.path.join(RAWDIR, p)
    os.makedirs(pdir, exist_ok=True)
    batches = make_batches(mf["file_id"].tolist(), BATCH)
    for i, chunk in enumerate(batches):
        dest = os.path.join(pdir, f"batch_{i:03d}.tar.gz")
        print(f"  lo {i+1}/{len(batches)} ({len(chunk)} file) ...", end=" ", flush=True)
        download_batch(chunk, dest)
        print(f"{os.path.getsize(dest)/1e6:.1f} MB")

    tpm, gene_meta = assemble(p, mf)
    print(f"  TPM: {tpm.shape[0]} gene x {tpm.shape[1]} file")
    tcga_raw[p] = {"manifest": mf, "tpm": tpm, "gene_meta": gene_meta}

In [ ]:
# Hai project dung cung bang gene GENCODE -> 1 GENE_META chung
GENE_META = tcga_raw[PROJECTS[0]]["gene_meta"]
for p in PROJECTS[1:]:
    assert set(tcga_raw[p]["gene_meta"].index) == set(GENE_META.index), f"{p}: bang gene khac nhau"

mf_all = pd.concat([d["manifest"].assign(project=p) for p, d in tcga_raw.items()], ignore_index=True)
mf_all["patient_id"] = mf_all["case_id"]
mf_all["barcode_type_code"] = mf_all["sample_id"].astype(str).str[13:15]
mf_all["is_ffpe"] = mf_all["is_ffpe"].astype(str).str.lower().eq("true")
mf_all["has_expr"] = [fid in tcga_raw[p]["tpm"].columns for fid, p in zip(mf_all["file_id"], mf_all["project"])]

mismatch = mf_all["patient_id"] != mf_all["sample_id"].astype(str).str[:12]
assert not mismatch.any(), f"case_id khong khop barcode: {mf_all.loc[mismatch, ['case_id', 'sample_id']].head()}"

proj_per_patient = mf_all.groupby("patient_id")["project"].nunique()
CROSS_PROJECT_PATIENTS = sorted(proj_per_patient.index[proj_per_patient > 1])
print("Benh nhan xuat hien o ca 2 project:", CROSS_PROJECT_PATIENTS)

# Ma loai mau trong barcode phai khop sample_type -> dung de doi chieu
SAMPLE_TYPE_CODE = {"Primary Tumor": "01", "Solid Tissue Normal": "11"}
display(pd.crosstab([mf_all["project"], mf_all["sample_type"]], mf_all["has_expr"]))


def pick_one_per_patient(df, keys):
    # Moi (benh nhan, lop) giu 1 mau: uu tien khong FFPE, roi sample_id / file_id nho nhat (vd 01A truoc 01B)
    return (df.sort_values(keys + ["is_ffpe", "sample_id", "file_id"])
              .drop_duplicates(keys, keep="first"))


def tcga_expression(sel):
    # sel: index sample_id, cot file_id + project -> sample x gene, log2(TPM+1), chi protein_coding
    cols = []
    for p, g in sel.groupby("project"):
        t = tcga_raw[p]["tpm"].loc[GENE_META.index, g["file_id"]]
        t.columns = g.index
        cols.append(t)
    tpm = pd.concat(cols, axis=1)[sel.index]
    pc = GENE_META["gene_type"] == "protein_coding"
    expr = np.log2(tpm.loc[pc] + 1)
    X = collapse_by_symbol(expr, GENE_META.loc[pc, "gene_name"]).T.astype(np.float32)
    X.index.name, X.columns.name = "sample_id", None
    return X


TCGA_META_COLS = ["patient_id", "class", "label", "project", "sample_type", "file_id",
                  "is_ffpe", "primary_diagnosis", "stage_raw", "stage_group", "ajcc_edition",
                  "path_t", "path_n", "path_m", "stage_conflict", "stage_rule",
                  "stage_diagnosis_submitter_id", "diagnosis_mismatch"]

### 4.1. `tcga_luad_lusc_primary_tumor` — LUAD vs LUSC

1. Mọi file STAR - Counts của TCGA-LUAD và TCGA-LUSC.
2. Chỉ file đã tải được biểu hiện.
3. **Chỉ mẫu Primary Tumor** (`sample_type == "Primary Tumor"` và mã barcode `01`).
   Loại mô thường, Recurrent Tumor, Metastatic, Additional - New Primary.
4. Loại bệnh nhân xuất hiện ở cả hai project (nếu có).
5. **1 mẫu u / bệnh nhân** (ưu tiên không FFPE), để không có bệnh nhân nào đóng góp
   nhiều mẫu gần như trùng nhau.

Nhãn: `LUAD = 0`, `LUSC = 1` (theo project). Không yêu cầu nhãn giai đoạn.

In [ ]:
LL_LABELS = {"LUAD": 0, "LUSC": 1}
ll_log = FilterLog("tcga_luad_lusc_primary_tumor")
m = mf_all.copy()
ll_log.add("0_gdc_star_counts", "Moi file STAR - Counts cua TCGA-LUAD va TCGA-LUSC", m)
m = m[m["has_expr"]]
ll_log.add("1_co_bieu_hien", "Chi file da tai duoc ma tran bieu hien", m)
m = m[(m["sample_type"] == "Primary Tumor") & (m["barcode_type_code"] == SAMPLE_TYPE_CODE["Primary Tumor"])]
ll_log.add("2_chi_primary_tumor",
           "Chi mau Primary Tumor (sample_type = 'Primary Tumor' va ma barcode 01); "
           "loai mo thuong, Recurrent, Metastatic, Additional - New Primary", m)
m = m[~m["patient_id"].isin(CROSS_PROJECT_PATIENTS)]
ll_log.add("3_loai_benh_nhan_2_project", "Loai benh nhan co mau o ca TCGA-LUAD va TCGA-LUSC", m)
m = pick_one_per_patient(m, ["patient_id"])
ll_log.add("4_mot_mau_moi_benh_nhan", "Giu 1 mau u / benh nhan (uu tien khong FFPE, roi sample_id nho nhat)", m)

m = m.set_index("sample_id")
m["class"] = m["project"].str.replace("TCGA-", "", regex=False)
m["label"] = m["class"].map(LL_LABELS).astype(int)
assert m.index.is_unique and m["patient_id"].is_unique
assert (m["sample_type"] == "Primary Tumor").all()

ll_X = tcga_expression(m)
ll_meta = m.loc[ll_X.index, TCGA_META_COLS]
print(f"\nMa tran: {ll_X.shape} | lop:", ll_meta["class"].value_counts().to_dict())
display(ll_log.table())

### 4.2. `tcga_normal_vs_tumor_sanity` — Normal vs u

1. Mọi file STAR - Counts của TCGA-LUAD và TCGA-LUSC.
2. Chỉ file đã tải được biểu hiện.
3. **Chỉ Solid Tissue Normal (mã 11) và Primary Tumor (mã 01)** — không dùng dữ liệu
   ngoài TCGA (không có RNA tham chiếu hay mẫu GEO).
4. Loại bệnh nhân xuất hiện ở cả hai project (nếu có).
5. 1 mẫu / (bệnh nhân, lớp): mỗi bệnh nhân có tối đa 1 mẫu u và 1 mẫu thường.

Nhãn: `Normal = 0`, `Tumor = 1`. Phân tầng theo `lớp × project`; nhóm theo `patient_id`
nên **cặp u–thường của cùng một bệnh nhân luôn nằm trong cùng một phần / cùng một tập**.

In [ ]:
NT_LABELS = {"Normal": 0, "Tumor": 1}
NT_TYPES = {"Solid Tissue Normal": "Normal", "Primary Tumor": "Tumor"}
nt_log = FilterLog("tcga_normal_vs_tumor_sanity")
m = mf_all.copy()
nt_log.add("0_gdc_star_counts", "Moi file STAR - Counts cua TCGA-LUAD va TCGA-LUSC", m)
m = m[m["has_expr"]]
nt_log.add("1_co_bieu_hien", "Chi file da tai duoc ma tran bieu hien", m)
m = m[m["sample_type"].isin(NT_TYPES) & (m["barcode_type_code"] == m["sample_type"].map(SAMPLE_TYPE_CODE))]
nt_log.add("2_chi_normal_va_primary_tumor",
           "Chi Solid Tissue Normal (ma 11) va Primary Tumor (ma 01) cua TCGA-LUAD/LUSC", m)
m = m[~m["patient_id"].isin(CROSS_PROJECT_PATIENTS)]
nt_log.add("3_loai_benh_nhan_2_project", "Loai benh nhan co mau o ca TCGA-LUAD va TCGA-LUSC", m)
m = m.assign(**{"class": m["sample_type"].map(NT_TYPES)})
m = pick_one_per_patient(m, ["patient_id", "class"])
nt_log.add("4_mot_mau_moi_benh_nhan_moi_lop",
           "Giu 1 mau / (benh nhan, lop): toi da 1 mau u va 1 mau thuong moi benh nhan", m)

m = m.set_index("sample_id")
m["label"] = m["class"].map(NT_LABELS).astype(int)
m["strata"] = m["class"] + "|" + m["project"]
assert m.index.is_unique

nt_X = tcga_expression(m)
nt_meta = m.loc[nt_X.index, TCGA_META_COLS + ["strata"]]
n_pair = int((nt_meta.groupby("patient_id")["class"].nunique() == 2).sum())
print(f"\nMa tran: {nt_X.shape} | {n_pair} benh nhan co ca mau u va mau thuong")
display(pd.crosstab(nt_meta["project"], nt_meta["class"], margins=True))
display(nt_log.table())

## 5. Chia fold theo bệnh nhân và thống kê lớp

`StratifiedGroupKFold(groups = patient_id)` chia mỗi bộ thành 5 phần P1–P5; thử
`SEED_SEARCH` seed và chọn cách chia có tổng độ lệch số mẫu mỗi tầng so với mức lý
tưởng (`tổng tầng / 5`) nhỏ nhất. Seed được chọn ghi vào `config.json`.

`assert` kiểm tra: mỗi bệnh nhân nằm trọn trong 1 phần; trong từng fold, tập bệnh
nhân của train/val/test đôi một rời nhau; qua 5 fold mỗi mẫu làm test đúng 1 lần
và val đúng 1 lần.

`class_stats` đếm **số bệnh nhân và số mẫu theo lớp** cho: toàn bộ (`all`), từng phần
(`part`), và từng tập train/val/test của từng fold.

In [ ]:
def make_parts(meta, strat_col, n_splits=N_FOLDS, seeds=SEED_SEARCH):
    # Tra ve (so phan 1..n_splits cua tung mau, seed duoc chon, do lech)
    y = pd.factorize(meta[strat_col])[0]
    groups = meta["patient_id"].values
    ideal = pd.Series(y).value_counts().sort_index() / n_splits
    best_part, best_score, best_seed = None, np.inf, None
    for s in seeds:
        sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=s)
        part = np.empty(len(y), dtype=int)
        for k, (_, idx) in enumerate(sgkf.split(np.zeros(len(y)), y, groups), start=1):
            part[idx] = k
        ct = pd.crosstab(part, y).reindex(columns=ideal.index, fill_value=0)
        score = float((ct - ideal.values).abs().values.sum())
        if score < best_score:
            best_part, best_score, best_seed = part, score, s
    print(f"  seed chon: {best_seed} | tong do lech so mau/tang: {best_score:.1f}")
    return best_part, best_seed, best_score


def fold_split(meta, r):
    # Vai tro (train/val/test) cua tung mau trong fold r
    plan = FOLD_PLAN[r]
    role = {p: "train" for p in plan["train"]}
    role[plan["val"]], role[plan["test"]] = "val", "test"
    return meta["part"].map(role)


def verify_no_patient_leak(meta):
    assert (meta.groupby("patient_id")["part"].nunique() == 1).all(), "Co benh nhan nam o nhieu phan"
    for r in FOLD_PLAN:
        split = meta[f"fold_{r}"]
        assert split.isin(["train", "val", "test"]).all()
        pats = {sp: set(meta.loc[split == sp, "patient_id"]) for sp in ["train", "val", "test"]}
        for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
            inter = pats[a] & pats[b]
            assert not inter, f"Fold {r}: benh nhan o ca {a} va {b}: {sorted(inter)[:5]}"
    roles = meta[[f"fold_{r}" for r in FOLD_PLAN]]
    assert ((roles == "test").sum(axis=1) == 1).all() and ((roles == "val").sum(axis=1) == 1).all()
    n_multi = int((meta.groupby("patient_id").size() > 1).sum())
    print(f"  OK: {meta['patient_id'].nunique()} benh nhan, {N_FOLDS} fold khong ro ri giua train/val/test "
          f"({n_multi} benh nhan co >1 mau, deu nam tron trong 1 phan)")


def class_stats(meta, classes):
    # Bang dai: fold | split | class | n_patients | n_samples (class = 'TONG' la tong cac lop)
    views = [("all", "all", pd.Series(True, index=meta.index))]
    views += [("part", f"P{k}", meta["part"] == k) for k in range(1, N_FOLDS + 1)]
    views += [(f"fold_{r}", sp, meta[f"fold_{r}"] == sp) for r in FOLD_PLAN for sp in ["train", "val", "test"]]
    rows = []
    for fold, split, mask in views:
        sub = meta[mask]
        for c in list(classes) + ["TONG"]:
            s = sub if c == "TONG" else sub[sub["class"] == c]
            rows.append({"fold": fold, "split": split, "class": c,
                         "n_patients": int(s["patient_id"].nunique()), "n_samples": int(len(s))})
    return pd.DataFrame(rows)


def stats_wide(stats):
    # 'so mau (so benh nhan)' theo lop, moi dong la 1 (fold, split)
    s = stats.assign(cell=stats["n_samples"].astype(str) + " (" + stats["n_patients"].astype(str) + ")")
    order = list(dict.fromkeys(zip(stats["fold"], stats["split"])))
    w = s.pivot(index=["fold", "split"], columns="class", values="cell")
    return w.loc[order, list(dict.fromkeys(stats["class"]))]


def warn_missing_classes(stats):
    miss = stats[(stats["fold"].str.startswith("fold_")) & (stats["split"] != "train")
                 & (stats["class"] != "TONG") & (stats["n_samples"] == 0)]
    if len(miss):
        print("  CANH BAO: lop vang mat trong val/test:", miss[["fold", "split", "class"]].values.tolist())

## 6. Lưu 3 bộ dữ liệu

Mỗi bộ được dựng trong thư mục tạm, đóng thành `<OUTDIR>/<bo>.zip`, đồng thời chép
các file nhỏ (`config.json`, `DATA_CARD.md`, `fold_assignment.csv`, `class_stats.csv`,
`filter_log.csv`) ra `<OUTDIR>/<bo>/` để xem nhanh trên Drive.

Đọc lại:

```python
X = pd.read_hdf("expression.h5", key="expression")                 # mau x gene
fa = pd.read_csv("fold_assignment.csv", index_col="sample_id")
train = X.loc[fa.index[fa["fold_1"] == "train"]]; y_train = fa.loc[train.index, "label"]
```

In [ ]:
SMALL_FILES = ["config.json", "DATA_CARD.md", "fold_assignment.csv", "class_stats.csv", "filter_log.csv"]


def build_config(name, spec, meta, X, log, seed, score, files):
    cls = meta["class"].value_counts()
    return {
        "dataset": name,
        "task": spec["task"],
        "notebook": NOTEBOOK,
        "created_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "commit_hash": COMMIT_HASH,
        "commit_source": COMMIT_SOURCE,
        "seed": SEED,
        "seed_search": {"start": SEED_SEARCH[0], "stop": SEED_SEARCH[-1], "n": len(SEED_SEARCH)},
        "selected_seed": int(seed),
        "split_imbalance_score": score,
        "split_method": (f"StratifiedGroupKFold(n_splits={N_FOLDS}, shuffle=True, random_state=selected_seed), "
                         f"groups=patient_id -> phan P1..P{N_FOLDS}; fold r: test=P_r, val=P_(r+1), train=con lai"),
        "group_col": "patient_id",
        "stratify_by": spec["stratify_desc"],
        "fold_plan": {f"fold_{r}": {"train_parts": p["train"], "val_part": p["val"], "test_part": p["test"]}
                      for r, p in FOLD_PLAN.items()},
        "label_col": "label",
        "label_mapping": spec["labels"],
        "inclusion_criteria": [r["tieu_chi"] for r in log.rows[1:]],
        "sample_counts": log.rows,
        "n_samples": int(len(meta)),
        "n_patients": int(meta["patient_id"].nunique()),
        "n_genes": int(X.shape[1]),
        "class_counts": {c: {"n_samples": int(cls.get(c, 0)),
                             "n_patients": int(meta.loc[meta["class"] == c, "patient_id"].nunique())}
                         for c in spec["labels"]},
        "expression": spec["expression"],
        "source": spec["source"],
        "versions": VERSIONS,
        "files": files,
    }


def data_card(name, spec, cfg, log, stats):
    labels = pd.DataFrame([{"label": v, "class": k,
                            "n_samples": cfg["class_counts"][k]["n_samples"],
                            "n_patients": cfg["class_counts"][k]["n_patients"]}
                           for k, v in spec["labels"].items()])
    flog = log.table()[["step", "buoc", "tieu_chi", "n_samples", "n_patients", "n_removed"]]
    wide = stats_wide(stats).reset_index()
    plan = pd.DataFrame([{"fold": f"fold_{r}", "train": "+".join(f"P{x}" for x in p["train"]),
                          "val": f"P{p['val']}", "test": f"P{p['test']}"} for r, p in FOLD_PLAN.items()])
    lines = [
        f"# Data card: `{name}`", "",
        f"**Tác vụ:** {spec['task']}", "",
        f"- Nguồn: {spec['source']['description']}",
        f"- Số mẫu / bệnh nhân / gene: {cfg['n_samples']} / {cfg['n_patients']} / {cfg['n_genes']}",
        f"- Ngày tạo (UTC): {cfg['created_at_utc']}",
        f"- Commit: `{cfg['commit_hash']}` ({cfg['commit_source']}) — notebook `{NOTEBOOK}`",
        f"- Seed gốc: {SEED}; seed chia được chọn: {cfg['selected_seed']} "
        f"(tìm trong {SEED_SEARCH[0]}..{SEED_SEARCH[-1]})",
        "",
        "## Tiêu chí lọc mẫu", "",
        *[f"{i}. {c}" for i, c in enumerate(spec["criteria_vi"], 1)],
        "",
        "## Số mẫu trước / sau từng bước lọc", "",
        md_table(flog), "",
        "## Nhãn", "",
        md_table(labels), "",
        "## Ma trận biểu hiện", "",
        *[f"- {x}" for x in spec["expression"]["description_vi"]],
        "",
        "## Chia fold theo bệnh nhân", "",
        f"- `StratifiedGroupKFold` với `groups = patient_id`, phân tầng theo {spec['stratify_desc']}.",
        "- Mỗi bệnh nhân nằm trọn trong một phần; trong mỗi fold, bệnh nhân của train/val/test "
        "đôi một rời nhau (đã kiểm tra bằng `assert`).",
        "", md_table(plan), "",
        "Số mẫu (số bệnh nhân) theo lớp:", "",
        md_table(wide), "",
        "## Tệp", "",
        "- `expression.h5` (key `expression`): mẫu × gene, float32, index `sample_id`.",
        "- `metadata.csv`: metadata đã lọc của các mẫu được giữ.",
        "- `fold_assignment.csv`: `part` và vai trò train/val/test của từng mẫu trong `fold_1..fold_5`.",
        "- `class_stats.csv`: số bệnh nhân và số mẫu theo lớp trong từng phần / từng tập.",
        "- `filter_log.csv`: số mẫu trước / sau từng bước lọc.",
        "- `config.json`: seed, commit, tiêu chí lọc, ánh xạ nhãn, phiên bản thư viện.",
    ]
    if EXPORT_FOLD_FILES:
        lines.append("- `folds/fold_<r>/{train,val,test}." + "|".join(FOLD_FILE_FORMATS)
                     + "`: bảng mẫu × (gene..., `label`) dựng sẵn từ `fold_assignment.csv`.")
    lines += ["", "## Lưu ý và hạn chế", "", *[f"- {x}" for x in spec["caveats_vi"]], ""]
    return "\n".join(lines)


def save_dataset(name, X, meta, log, spec):
    print(f"\n=== {name}: {X.shape[0]} mau x {X.shape[1]} gene ===")
    assert X.index.equals(meta.index), "X va meta lech index"
    assert meta["label"].notna().all() and meta["patient_id"].notna().all()
    meta = meta.copy()
    meta["part"], seed, score = make_parts(meta, spec["stratify_col"])
    for r in FOLD_PLAN:
        meta[f"fold_{r}"] = fold_split(meta, r)
    verify_no_patient_leak(meta)
    stats = class_stats(meta, list(spec["labels"]))
    warn_missing_classes(stats)
    display(stats_wide(stats))

    ds = os.path.join(STAGEDIR, name)
    shutil.rmtree(ds, ignore_errors=True)
    os.makedirs(ds)
    fold_cols = [f"fold_{r}" for r in FOLD_PLAN]
    X.to_hdf(os.path.join(ds, "expression.h5"), key="expression", mode="w", format="fixed")
    meta.drop(columns=["part"] + fold_cols).to_csv(os.path.join(ds, "metadata.csv"))
    meta[spec["assignment_cols"] + ["part"] + fold_cols].to_csv(os.path.join(ds, "fold_assignment.csv"))
    stats.to_csv(os.path.join(ds, "class_stats.csv"), index=False)
    log.table().to_csv(os.path.join(ds, "filter_log.csv"), index=False)

    if EXPORT_FOLD_FILES:
        data = X.copy()
        data["label"] = meta["label"].astype(int)
        for fmt in FOLD_FILE_FORMATS:
            for r in FOLD_PLAN:
                fd = os.path.join(ds, "folds", f"fold_{r}")
                os.makedirs(fd, exist_ok=True)
                for sp in ["train", "val", "test"]:
                    part = data.loc[meta.index[meta[f"fold_{r}"] == sp]]
                    path = os.path.join(fd, f"{sp}.{fmt}")
                    if fmt == "h5":
                        part.to_hdf(path, key="data", mode="w", format="fixed")
                    else:
                        part.to_csv(path)

    files = sorted(os.path.relpath(os.path.join(dp, f), ds).replace(os.sep, "/")
                   for dp, _, fs in os.walk(ds) for f in fs) + ["config.json", "DATA_CARD.md"]
    cfg = build_config(name, spec, meta, X, log, seed, score, sorted(files))
    with open(os.path.join(ds, "config.json"), "w", encoding="utf-8") as f:
        json.dump(cfg, f, indent=2, ensure_ascii=False)
    with open(os.path.join(ds, "DATA_CARD.md"), "w", encoding="utf-8") as f:
        f.write(data_card(name, spec, cfg, log, stats))

    zip_path = os.path.join(OUTDIR, f"{name}.zip")
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for rel in cfg["files"]:
            zf.write(os.path.join(ds, rel), arcname=f"{name}/{rel}")
    small_dir = os.path.join(OUTDIR, name)
    os.makedirs(small_dir, exist_ok=True)
    for fname in SMALL_FILES:
        shutil.copy2(os.path.join(ds, fname), os.path.join(small_dir, fname))
    print(f"  -> {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB) + {small_dir}/")
    return zip_path, cfg

### 6.1. Đặc tả từng bộ (tiêu chí lọc, nhãn, hạn chế) và lưu

In [ ]:
GDC_RELEASE = GDC_STATUS.get("data_release", "khong ro")
TCGA_EXPRESSION = {
    "unit": "log2(TPM + 1)", "workflow": "GDC STAR - Counts, cot tpm_unstranded",
    "gene_filter": "gene_type == protein_coding", "gene_id": "gene_name (trung ten -> giu gene phuong sai lon nhat)",
    "scaling": "khong",
    "description_vi": ["Giá trị `log2(TPM + 1)` từ cột `tpm_unstranded` của GDC STAR - Counts.",
                       "Chỉ gene `protein_coding`; Ensembl ID đổi sang `gene_name`, trùng tên → giữ gene có phương sai lớn nhất "
                       "(tính trên các mẫu của bộ này, không dùng nhãn).",
                       "Chưa chuẩn hoá / scale; chưa hiệu chỉnh batch."],
}
TCGA_SOURCE = {"description": f"GDC API, TCGA-LUAD + TCGA-LUSC, STAR - Counts (GDC data release: {GDC_RELEASE})",
               "projects": PROJECTS, "api": API, "gdc_status": GDC_STATUS,
               "cross_project_patients_removed": CROSS_PROJECT_PATIENTS}

SPECS = {
    "gse68465_stage_tumor_only": {
        "task": "Phân loại giai đoạn bệnh (AJCC I/II/III/IV) trên mẫu u LUAD",
        "labels": {k: v for k, v in GSE_STAGE_LABELS.items()},
        "stratify_col": "class", "stratify_desc": "lớp giai đoạn",
        "assignment_cols": ["patient_id", "class", "label", "stage_raw"],
        "criteria_vi": [
            f"Bắt đầu từ toàn bộ mẫu trong series matrix {GSE} ({GSE_GPL}).",
            "**Loại vĩnh viễn 19 mẫu RNA tham chiếu Stratagene** (`source_name` chứa `Stratagene`). "
            "Bản cũ gán nhãn `Normal` cho các mẫu này, nhưng đây là RNA thương mại dùng làm chuẩn, không phải mô "
            "của bệnh nhân. Bộ dữ liệu này không có lớp Normal.",
            "Chỉ giữ mẫu u LUAD: `disease_state` chứa `Adenocarcinoma`.",
            "Chỉ giữ mẫu có giai đoạn hợp lệ I/II/III/IV, suy ra từ `disease_stage` (pT/pN, giả định M0). "
            "Mẫu thiếu giai đoạn hoặc giai đoạn 0 bị loại.",
        ],
        "expression": {"unit": "log2(MAS5)" if GSE_LOG2_APPLIED else "gia tri goc trong series matrix",
                       "platform": GSE_GPL, "probe_to_gene": f"{GSE_GPL}.annot.gz, bo probe anh xa nhieu gene (///)",
                       "collapse": "nhieu probe cung gene -> giu probe phuong sai lon nhat", "scaling": "khong",
                       "description_vi": [
                           f"Giá trị MAS5 từ series matrix, {'đã chuyển sang log2(x + 1)' if GSE_LOG2_APPLIED else 'giữ nguyên thang đo gốc'}.",
                           f"Probe → gene symbol theo `{GSE_GPL}.annot.gz`; bỏ probe ánh xạ tới nhiều gene (`A///B`); "
                           "nhiều probe cùng gene → giữ probe có phương sai lớn nhất (không dùng nhãn).",
                           "Chưa chuẩn hoá / scale; chưa hiệu chỉnh batch giữa các trung tâm."]},
        "source": {"description": f"GEO {GSE} (series matrix, {GSE_GPL})", "accession": GSE,
                   "urls": GSE_URLS + [GSE_ANNOT_URL],
                   "removed_stratagene_reference_gsm": GSE_REMOVED_REFERENCE},
        "caveats_vi": [
            "Nhãn giai đoạn được suy ra từ pT/pN với giả định M0 (dữ liệu không ghi pM), nên lớp IV "
            "có thể không xuất hiện; ánh xạ nhãn vẫn giữ IV = 3 để cố định.",
            "GEO không có mã bệnh nhân riêng: mỗi mẫu u được coi là một bệnh nhân (`patient_id = GSM`), "
            "trừ khi metadata có trường mã bệnh nhân.",
            "Ánh xạ nhãn khác bản cũ: bản cũ dùng `Normal=0, I=1, ...`; bản này `I=0, II=1, III=2, IV=3`.",
            "Dữ liệu đa trung tâm (Director's Challenge), chưa hiệu chỉnh batch. Lớp I chiếm đa số → nên báo "
            "balanced accuracy / macro-F1.",
        ],
    },
    "tcga_luad_lusc_primary_tumor": {
        "task": "Phân loại mô học LUAD vs LUSC trên mẫu u nguyên phát",
        "labels": LL_LABELS,
        "stratify_col": "class", "stratify_desc": "lớp (LUAD/LUSC)",
        "assignment_cols": ["patient_id", "class", "label", "project"],
        "criteria_vi": [
            "Bắt đầu từ mọi file GDC STAR - Counts của TCGA-LUAD và TCGA-LUSC.",
            "Chỉ file đã tải được ma trận biểu hiện.",
            "**Chỉ mẫu Primary Tumor** (`sample_type = Primary Tumor`, mã barcode `01`). Loại Solid Tissue Normal, "
            "Recurrent Tumor, Metastatic, Additional - New Primary.",
            "Loại bệnh nhân có mẫu ở cả hai project (nếu có).",
            "Giữ 1 mẫu u / bệnh nhân: ưu tiên mẫu không FFPE, sau đó `sample_id` nhỏ nhất (vd `01A` trước `01B`).",
            "Nhãn lấy theo project: TCGA-LUAD → LUAD, TCGA-LUSC → LUSC. Không yêu cầu nhãn giai đoạn.",
        ],
        "expression": TCGA_EXPRESSION, "source": TCGA_SOURCE,
        "caveats_vi": [
            "Nhãn theo project TCGA, không xét lại `primary_diagnosis` (lấy từ bản ghi u chỉ điểm, giữ trong "
            "`metadata.csv` để tham khảo); `diagnosis_mismatch = True` đánh dấu ca chẩn đoán không khớp project, "
            "dùng cho phân tích độ nhạy.",
            "Giai đoạn (`stage_raw`, `stage_group`, `ajcc_edition`, `path_t/n/m`) chỉ dùng cho bài toán phụ; chọn "
            "theo quy tắc của `select_diagnosis` (giống `scripts/patch_v2_tcga_stage.py`). `stage_conflict = True`: "
            "các bản ghi diagnoses có giai đoạn khác nhau, đã lấy giai đoạn của u chỉ điểm.",
            "Mỗi bệnh nhân chỉ có 1 mẫu nên chia theo bệnh nhân tương đương chia theo mẫu; vẫn dùng `groups` để nhất quán.",
            "Có thể có hiệu ứng trung tâm thu mẫu (TSS) và batch; chưa hiệu chỉnh.",
        ],
    },
    "tcga_normal_vs_tumor_sanity": {
        "task": "Sanity check: phân biệt mô phổi thường và mô u (Normal vs Tumor) trong TCGA-LUAD/LUSC",
        "labels": NT_LABELS,
        "stratify_col": "strata", "stratify_desc": "lớp × project (Normal/Tumor × LUAD/LUSC)",
        "assignment_cols": ["patient_id", "class", "label", "project"],
        "criteria_vi": [
            "Bắt đầu từ mọi file GDC STAR - Counts của TCGA-LUAD và TCGA-LUSC.",
            "Chỉ file đã tải được ma trận biểu hiện.",
            "**Chỉ Solid Tissue Normal (mã `11`) và Primary Tumor (mã `01`)** của TCGA-LUAD/LUSC. Không dùng "
            "mẫu ngoài TCGA (không có RNA tham chiếu Stratagene hay mẫu GEO).",
            "Loại bệnh nhân có mẫu ở cả hai project (nếu có).",
            "Giữ 1 mẫu / (bệnh nhân, lớp): mỗi bệnh nhân có tối đa 1 mẫu u và 1 mẫu thường "
            "(ưu tiên không FFPE, sau đó `sample_id` nhỏ nhất).",
        ],
        "expression": TCGA_EXPRESSION, "source": TCGA_SOURCE,
        "caveats_vi": [
            "Đây là tác vụ kiểm tra pipeline, rất dễ (u và mô thường khác biệt mạnh); không dùng làm kết quả chính.",
            "Đa số mẫu Normal có mẫu u ghép cặp từ cùng bệnh nhân; chia theo `patient_id` bảo đảm cặp này luôn "
            "nằm cùng một phần, nên không có rò rỉ bệnh nhân giữa train/val/test.",
            "Lớp Normal ít hơn nhiều so với Tumor → nên báo balanced accuracy / macro-F1.",
        ],
    },
}

DATASETS = {
    "gse68465_stage_tumor_only": (gse_X, gse_meta, gse_log),
    "tcga_luad_lusc_primary_tumor": (ll_X, ll_meta, ll_log),
    "tcga_normal_vs_tumor_sanity": (nt_X, nt_meta, nt_log),
}

results = {}
for name, (X, meta, log) in DATASETS.items():
    results[name] = save_dataset(name, X, meta, log, SPECS[name])

In [ ]:
# Tong hop ca lan chay -> <OUTDIR>/run_summary.json
summary = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "run_started_utc": RUN_STARTED_UTC,
    "commit_hash": COMMIT_HASH, "commit_source": COMMIT_SOURCE, "notebook": NOTEBOOK,
    "seed": SEED, "versions": VERSIONS,
    "datasets": {name: {"zip": os.path.basename(zp), "task": cfg["task"], "selected_seed": cfg["selected_seed"],
                        "n_samples": cfg["n_samples"], "n_patients": cfg["n_patients"], "n_genes": cfg["n_genes"],
                        "class_counts": cfg["class_counts"],
                        "samples_before_after": [cfg["sample_counts"][0]["n_samples"], cfg["n_samples"]]}
                 for name, (zp, cfg) in results.items()},
}
with open(os.path.join(OUTDIR, "run_summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

display(pd.DataFrame({name: {"mau truoc loc": d["samples_before_after"][0], "mau sau loc": d["n_samples"],
                             "benh nhan": d["n_patients"], "gene": d["n_genes"], "seed chon": d["selected_seed"],
                             **{f"lop {c}": f"{v['n_samples']} ({v['n_patients']})" for c, v in d["class_counts"].items()}}
                      for name, d in summary["datasets"].items()}).T)

## 7. Kiểm tra lại 3 file zip

Đọc lại từ zip (không dùng biến trong bộ nhớ): đủ file, `expression.h5` khớp
`fold_assignment.csv`, không rò rỉ bệnh nhân, và các ràng buộc riêng của từng bộ.

In [ ]:
REQUIRED = ["expression.h5", "metadata.csv", "fold_assignment.csv", "class_stats.csv",
            "filter_log.csv", "config.json", "DATA_CARD.md"]
check_dir = os.path.join(WORKDIR, "datasets_v2_check")
shutil.rmtree(check_dir, ignore_errors=True)

for name, (zp, _) in results.items():
    with zipfile.ZipFile(zp) as zf:
        names = set(zf.namelist())
        missing = [f for f in REQUIRED if f"{name}/{f}" not in names]
        assert not missing, f"{name}: thieu {missing}"
        for f in REQUIRED:
            zf.extract(f"{name}/{f}", check_dir)
    d = os.path.join(check_dir, name)
    X = pd.read_hdf(os.path.join(d, "expression.h5"), key="expression")
    fa = pd.read_csv(os.path.join(d, "fold_assignment.csv"), index_col="sample_id")
    md_ = pd.read_csv(os.path.join(d, "metadata.csv"), index_col="sample_id")
    cfg = json.load(open(os.path.join(d, "config.json"), encoding="utf-8"))
    flog = pd.read_csv(os.path.join(d, "filter_log.csv"))

    assert list(X.index) == list(fa.index) == list(md_.index), f"{name}: index lech"
    assert "label" not in X.columns and X.shape[1] == cfg["n_genes"]
    assert flog["n_samples"].iloc[-1] == len(X) == cfg["n_samples"]
    assert cfg["commit_hash"] == COMMIT_HASH and cfg["seed"] == SEED
    for r in FOLD_PLAN:
        pats = {sp: set(fa.loc[fa[f"fold_{r}"] == sp, "patient_id"]) for sp in ["train", "val", "test"]}
        assert not (pats["train"] & pats["val"] or pats["train"] & pats["test"] or pats["val"] & pats["test"])
    assert (fa.groupby("patient_id")["part"].nunique() == 1).all()

    if name == "gse68465_stage_tumor_only":
        assert not md_["source_name"].str.contains("stratagene", case=False).any()
        assert not set(GSE_REMOVED_REFERENCE) & set(X.index)
        assert md_["stage"].isin(VALID_STAGES).all() and "Normal" not in set(md_["class"])
        assert flog.loc[flog["buoc"] == "1_loai_stratagene", "n_removed"].item() == EXPECTED_STRATAGENE
    elif name == "tcga_luad_lusc_primary_tumor":
        assert (md_["sample_type"] == "Primary Tumor").all() and md_["patient_id"].is_unique
        assert set(md_["class"]) == {"LUAD", "LUSC"}
        # Chan loi diagnoses[0] cu (mat ~20% giai doan); GDC release 46 chi thieu 12/1018
        assert md_["stage_group"].notna().mean() > 0.95, "qua nhieu ca thieu giai doan"
    else:
        assert set(md_["sample_type"]) == {"Primary Tumor", "Solid Tissue Normal"}
        assert set(md_["project"]) <= set(PROJECTS)
        assert not md_.duplicated(["patient_id", "class"]).any()

    n_fold_files = sum(1 for n in names if "/folds/" in n)
    print(f"OK {name:32s} {X.shape[0]} mau / {fa['patient_id'].nunique()} benh nhan x {X.shape[1]} gene | "
          f"{n_fold_files} file fold | {os.path.getsize(zp)/1e6:.1f} MB")

print("\nDATA_CARD mau (gse68465_stage_tumor_only):\n")
print(open(os.path.join(check_dir, "gse68465_stage_tumor_only", "DATA_CARD.md"), encoding="utf-8").read())

## 8. (Tuỳ chọn) Ngắt kết nối Colab

Chỉ chạy sau khi đã thấy đủ 3 file zip và `run_summary.json` trên Drive.

In [ ]:
if IN_COLAB:
    print(os.listdir(OUTDIR))
    from google.colab import runtime
    runtime.unassign()